## BERT Sentiment Explainability

Why did BERT classify a review as positive, negative, or neutral? Which words contributed to the classification?  

To answer these questions, we will use model explainability techniques to analyze the model's predictions and understand the factors influencing its decisions.

To analyze the model's predictions, we will use the `captum` library through `transformers-interpret`, which provides `SequenceClassificationExplainer`, a tool that allows us to visualize the contribution of each word in the input text to the model's prediction. A simple visualization is also provided. This will help us understand which words were most influential in determining the sentiment of the review.

In [2]:
from pathlib import Path
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers_interpret import SequenceClassificationExplainer

In [3]:
MODEL_DIR = Path('../src/models/bert-reviews-tuned')
DATA_DIR  = Path('../data/processed')
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
 
cls_explainer = SequenceClassificationExplainer(model, tokenizer)
 
N_STEPS = 25 

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [4]:
def truncate_to_max_tokens(text, max_tokens=512):
    '''truncates the text not to conflict with BERT's limit'''
    token_ids = tokenizer.encode(text, max_length=max_tokens, truncation=True)
    return tokenizer.decode(token_ids, skip_special_tokens=True)

In [5]:
df_test = pd.read_csv(DATA_DIR / 'test_results.csv')
 
print(df_test.shape)

(300, 3)


For the explainability analysis, a few reviews were manually selected from the test set. Some of them are misclassified reviews which were analyzed in the previous section, other are well classified. It is just a sample, and it is not meant to be representative of the entire dataset. The goal is to provide insights into the model's decision-making process for specific examples.

In [8]:
selected = [164, 21, 272, 132, 268, 61]
 
for i in selected:
    text = truncate_to_max_tokens(df_test.loc[i, "text"])
    cls_explainer(text)
    print(f"Review {i}:")
    cls_explainer.visualize(true_class=df_test.loc[i, "label"])

Review 164:


Review 21:


Review 272:


Review 132:


Review 268:


Review 61:


The first three reviews are misclassified, while the last three are well classified.  
In the misclassified cases, we can clearly see that the model was influenced by certain words that led to an incorrect prediction.  

For example, in the review 272, the prediction was primarly caused by the word "disgusting", occurring a mistake, because it was not directly referred to the user's opinion, but his child's. Probably the strength of this word, made the model not consider other ligher words like "good" and "great", which have lower attribution.  

In the well-classified cases, we can observe that the model correctly identified the sentiment based on the presence of specific words, like "perfect" and "delicious" in positive reviews, and "stale" and "disappointed" in the negative one.